In [2]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

  Using cached qiskit-1.2.4-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached scipy-1.17.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached stevedore-5.7.0-py3-none-any.whl.metadata (2.4 kB)
  Using cached symengine-0.13.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.2 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached qiskit-1.2.4-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (4.8 MB)
Using cached symengine-0.13.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (49.7 MB)
Using cached dill-0.4.1-py3-none-any.whl (120 kB)
Using cached scipy-1.17.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (35.3 MB)
Using cached stevedore-5.7.0-py3-none-any.whl (54 kB)
Using cached sympy-1.14.0-py3-none-any.whl

In [3]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile 
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math 

# The aim of the assignment is to simulate the Ekert91 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.



In [4]:
# subroutines that are shared between both files are here to make it more readable

# Use a quantum circuit to generate a random bit
def random():
    q = QuantumCircuit(1)
    q.h(0)
    q.measure_all()
    backend = BasicSimulator()
    compiled = transpile(q, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result()
    counts = result_sim.get_counts(compiled)
    return counts.get("1",0)

# ^ from QC-Lab4-A-Complete.ipynb

# Generate 1 2 or 3
def randomChoice():
    q = QuantumCircuit(1)
    q.unitary([[1/math.sqrt(3), -math.sqrt(2/3)],[math.sqrt(2/3), 1/math.sqrt(3)]], [0])
    q.measure_all()

    backend = BasicSimulator()
    compiled = transpile(q, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result()
    counts = result_sim.get_counts(compiled)

    if counts.get("0",0) == 1:
        return 1
    else:
        if random() == 0:
            return 2
        else:
            return 3

# ^ slightly adjusted from QC-Lab4-A-Complete.ipynb

# Convert to 1 or -1
def pm(x):
    if x == 0:
        return 1
    else:
        return -1

# Prepare singlet state on an existing circuit
def singletCircuit(q):
    q.x(1)
    q.h(0)
    q.cx(0,1)
    q.z(0)

# alice measurement bases 1=X 2=W 3=Z
def measureAlice(q, basis):
    if basis == 1:
        q.h(0)
    elif basis == 2:
        q.ry(-math.pi/4,0)

# bob measurement bases 1=W 2=Z 3=V
def measureBob(q, basis):
    if basis == 1:
        q.ry(-math.pi/4,1)
    elif basis == 3:
        q.ry(math.pi/4,1)

In [13]:
# Simulation of the Ekert91 key distribution protocol without an attacker

def measurePair(aBasis,bBasis):
    q = QuantumCircuit(2)
    singletCircuit(q)

    measureAlice(q,aBasis)
    measureBob(q,bBasis)

    q.measure_all()

    backend = BasicSimulator()
    compiled = transpile(q, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result()
    counts = result_sim.get_counts(compiled)

    result = list(counts.keys())[0]

    aliceBit = int(result[1])
    bobBit = int(result[0])

    return [aliceBit,bobBit]

N = 100

aliceBases = []
bobBases = []

aliceBits = []
bobBits = []

aliceKey = []
bobKey = []

totalXW = 0
countXW = 0

totalXV = 0
countXV = 0

totalZW = 0
countZW = 0

totalZV = 0
countZV = 0

while len(aliceKey) < N:
    a = randomChoice()
    b = randomChoice()

    bits = measurePair(a,b)

    x = bits[0]
    y = bits[1]

    aliceBases = aliceBases + [a]
    bobBases = bobBases + [b]

    aliceBits = aliceBits + [x]
    bobBits = bobBits + [y]

    if a == 2 and b == 1:
        aliceKey = aliceKey + [x]
        bobKey = bobKey + [1-y]

    elif a == 3 and b == 2:
        aliceKey = aliceKey + [x]
        bobKey = bobKey + [1-y]

    elif a == 1 and b == 1:
        totalXW = totalXW + pm(x)*pm(y)
        countXW = countXW + 1

    elif a == 1 and b == 3:
        totalXV = totalXV + pm(x)*pm(y)
        countXV = countXV + 1

    elif a == 3 and b == 1:
        totalZW = totalZW + pm(x)*pm(y)
        countZW = countZW + 1

    elif a == 3 and b == 3:
        totalZV = totalZV + pm(x)*pm(y)
        countZV = countZV + 1

print("Alice bases:",aliceBases)
print("Bob bases:  ",bobBases)

print("Alice bits: ",aliceBits)
print("Bob bits:   ",bobBits)

print("Alice key:",aliceKey)
print("Bob key:  ",bobKey)

EXW = totalXW/countXW
EXV = totalXV/countXV
EZW = totalZW/countZW
EZV = totalZV/countZV

S = abs(EXW - EXV + EZW + EZV)

# ⊗ √ symbols for copy pasting

print("<X ⊗ W> =",EXW)
print("<X ⊗ V> =",EXV)
print("<Z ⊗ W> =",EZW)
print("<Z ⊗ V> =",EZV)

print("S =",S)

print("Counts:")
print("XW:",countXW)
print("XV:",countXV)
print("ZW:",countZW)
print("ZV:",countZV)
print("Total rounds:",len(aliceBases))

Alice bases: [2, 2, 2, 1, 2, 3, 3, 1, 2, 1, 3, 3, 1, 3, 3, 1, 3, 3, 2, 1, 2, 3, 3, 2, 1, 3, 2, 3, 3, 1, 2, 3, 2, 1, 2, 1, 2, 1, 2, 1, 2, 2, 3, 2, 3, 3, 2, 2, 1, 2, 1, 3, 2, 2, 1, 2, 1, 3, 3, 3, 2, 2, 1, 2, 2, 2, 2, 1, 3, 3, 1, 3, 1, 2, 2, 2, 3, 1, 2, 1, 2, 3, 3, 2, 1, 2, 3, 3, 2, 1, 3, 3, 3, 2, 1, 1, 3, 2, 3, 3, 1, 3, 2, 2, 2, 1, 3, 2, 3, 3, 3, 1, 3, 3, 1, 1, 1, 1, 1, 2, 1, 2, 1, 1, 3, 2, 2, 3, 1, 3, 2, 3, 1, 3, 1, 3, 3, 1, 3, 1, 2, 3, 2, 2, 1, 3, 1, 3, 1, 2, 2, 3, 3, 2, 1, 1, 1, 3, 3, 3, 1, 3, 3, 1, 3, 1, 1, 1, 1, 3, 2, 2, 1, 3, 3, 1, 2, 2, 3, 3, 2, 1, 1, 1, 3, 3, 1, 1, 3, 3, 1, 3, 2, 2, 1, 3, 1, 2, 2, 1, 1, 2, 1, 2, 1, 1, 2, 3, 2, 2, 2, 1, 3, 2, 1, 2, 2, 2, 1, 2, 1, 1, 1, 2, 3, 3, 1, 2, 1, 1, 3, 3, 1, 3, 3, 1, 1, 1, 3, 2, 3, 2, 3, 3, 3, 2, 3, 3, 1, 2, 1, 3, 3, 1, 3, 3, 3, 3, 1, 1, 3, 3, 1, 2, 2, 2, 2, 1, 3, 3, 1, 1, 3, 3, 1, 1, 2, 2, 2, 3, 3, 3, 1, 3, 1, 3, 1, 1, 3, 3, 1, 2, 1, 3, 3, 1, 2, 2, 1, 2, 3, 2, 1, 1, 2, 3, 3, 3, 1, 3, 1, 2, 3, 2, 2, 2, 3, 1, 1, 2, 3, 3, 1, 2, 3, 2, 1, 1, 3,

For cases where alice and bob measured in the same basis the results were opposite as expected.

Bell parameter was calculated as approximately S=3.13

higher than allowed classical limit of S<=2.

Higher than max quantum of S=2sqrt(2)

Works as expected with no attacker detected.
